In [2]:
from transformers import AutoModelForTokenClassification, AutoTokenizer
import torch

model_path = "../models/biobert-ner-final"

model = AutoModelForTokenClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Model loaded!")
print("Device:", device)

Model loaded!
Device: cpu


In [ ]:
print("=== FALSE POSITIVES — Things wrongly labeled as Disease ===\n")

false_positives = []

for i, (true_seq, pred_seq) in enumerate(zip(all_true_labels, all_predictions)):
    for j, (true_label, pred_label) in enumerate(zip(true_seq, pred_seq)):
        if pred_label == "B-Disease" and true_label != "B-Disease":
            false_positives.append({
                "sentence_idx": i,
                "token_position": j,
                "true_label": true_label
            })

print(f"Total false positives: {len(false_positives)}")

=== FALSE POSITIVES — Things wrongly labeled as Disease ===

Total false positives: 121


In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer

PARQUET_BASE = "hf://datasets/ncbi_disease@refs/convert/parquet/ncbi_disease"
raw_dataset = load_dataset("parquet", data_files={
    "test": f"{PARQUET_BASE}/test/0000.parquet",
})

label_names = ["O", "B-Disease", "I-Disease"]

def tokenize_and_align_dataset(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        padding="max_length",
        max_length=512,
        truncation=True
    )
    all_aligned_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        aligned_labels = []
        previous_word_id = None
        for word_id in tokenized_inputs.word_ids(batch_index=i):
            if word_id is None:
                aligned_labels.append(-100)
            elif word_id != previous_word_id:
                aligned_labels.append(labels[word_id])
            else:
                aligned_labels.append(-100)
            previous_word_id = word_id
        all_aligned_labels.append(aligned_labels)
    tokenized_inputs["labels"] = all_aligned_labels
    return tokenized_inputs

tokenized_test = raw_dataset["test"].map(
    tokenize_and_align_dataset,
    batched=True,
    remove_columns=raw_dataset["test"].column_names
)

print("Test set preprocessed!")
print("Test samples:", len(tokenized_test))

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/941 [00:00<?, ? examples/s]

Test set preprocessed!
Test samples: 941


In [4]:
import numpy as np
from torch.utils.data import DataLoader
import torch

tokenized_test.set_format("torch")
dataloader = DataLoader(tokenized_test, batch_size=16)

all_predictions = []
all_true_labels = []

with torch.no_grad():
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
        labels = labels.numpy()

        for pred_seq, label_seq in zip(predictions, labels):
            true_seq, pred_seq_clean = [], []
            for pred, label in zip(pred_seq, label_seq):
                if label != -100:
                    true_seq.append(label_names[label])
                    pred_seq_clean.append(label_names[pred])
            all_true_labels.append(true_seq)
            all_predictions.append(pred_seq_clean)

print("Predictions complete!")
print("Total sequences predicted:", len(all_predictions))

Predictions complete!
Total sequences predicted: 941


In [5]:
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report

print("=== Overall Performance ===")
print(f"Precision: {precision_score(all_true_labels, all_predictions):.4f}")
print(f"Recall:    {recall_score(all_true_labels, all_predictions):.4f}")
print(f"F1:        {f1_score(all_true_labels, all_predictions):.4f}")
print("\n=== Detailed Report ===")
print(classification_report(all_true_labels, all_predictions))

=== Overall Performance ===
Precision: 0.8479
Recall:    0.9000
F1:        0.8732

=== Detailed Report ===
              precision    recall  f1-score   support

     Disease       0.85      0.90      0.87       960

   micro avg       0.85      0.90      0.87       960
   macro avg       0.85      0.90      0.87       960
weighted avg       0.85      0.90      0.87       960



In [6]:
print("=== FALSE NEGATIVES — Diseases the model MISSED ===\n")

missed_entities = []

for i, (true_seq, pred_seq) in enumerate(zip(all_true_labels, all_predictions)):
    # Find entity spans in true labels
    j = 0
    while j < len(true_seq):
        if true_seq[j] == "B-Disease":
            # Found start of entity
            entity_tokens = [j]
            k = j + 1
            while k < len(true_seq) and true_seq[k] == "I-Disease":
                entity_tokens.append(k)
                k += 1
            # Check if model predicted this correctly
            predicted_correctly = pred_seq[j] == "B-Disease"
            if not predicted_correctly:
                missed_entities.append({
                    "sentence_idx": i,
                    "token_positions": entity_tokens,
                    "predicted_as": pred_seq[j]
                })
            j = k
        else:
            j += 1

print(f"Total missed disease entities: {len(missed_entities)}")
print(f"Out of total disease entities in test set")

=== FALSE NEGATIVES — Diseases the model MISSED ===

Total missed disease entities: 82
Out of total disease entities in test set


In [7]:
print("=== FALSE POSITIVES — Things wrongly labeled as Disease ===\n")

false_positives = []

for i, (true_seq, pred_seq) in enumerate(zip(all_true_labels, all_predictions)):
    for j, (true_label, pred_label) in enumerate(zip(true_seq, pred_seq)):
        if pred_label == "B-Disease" and true_label != "B-Disease":
            false_positives.append({
                "sentence_idx": i,
                "token_position": j,
                "true_label": true_label
            })

print(f"Total false positives: {len(false_positives)}")

=== FALSE POSITIVES — Things wrongly labeled as Disease ===

Total false positives: 121


In [8]:
total_diseases = sum(seq.count("B-Disease") for seq in all_true_labels)
total_missed = len(missed_entities)
total_fp = len(false_positives)

print("=== Error Analysis Summary ===\n")
print(f"Total disease entities in test set: {total_diseases}")
print(f"Correctly identified:               {total_diseases - total_missed}")
print(f"Missed (False Negatives):           {total_missed}")
print(f"Wrong predictions (False Positives):{total_fp}")
print(f"\nMiss rate: {total_missed/total_diseases*100:.1f}%")
print(f"False positive rate: {total_fp/(total_fp + (total_diseases - total_missed))*100:.1f}%")

=== Error Analysis Summary ===

Total disease entities in test set: 960
Correctly identified:               878
Missed (False Negatives):           82
Wrong predictions (False Positives):121

Miss rate: 8.5%
False positive rate: 12.1%


In [9]:
from datasets import load_dataset

PARQUET_BASE = "hf://datasets/ncbi_disease@refs/convert/parquet/ncbi_disease"
raw_test = load_dataset("parquet", data_files={
    "test": f"{PARQUET_BASE}/test/0000.parquet"
})["test"]

print("=== Sample Missed Diseases ===\n")
shown = 0
for miss in missed_entities[:20]:
    idx = miss["sentence_idx"]
    positions = miss["token_positions"]
    words = raw_test[idx]["tokens"]
    true_tags = raw_test[idx]["ner_tags"]

    if all(p < len(words) for p in positions):
        missed_entity = " ".join([words[p] for p in positions])
        sentence = " ".join(words)
        print(f"Missed entity:  '{missed_entity}'")
        print(f"Predicted as:    {miss['predicted_as']}")
        print(f"Sentence:        {sentence[:120]}")
        print()
        shown += 1
    if shown >= 10:
        break

=== Sample Missed Diseases ===

Missed entity:  'clonal malignancy'
Predicted as:    O
Sentence:        By analysing tumour DNA from patients with sporadic T - cell prolymphocytic leukaemia ( T - PLL ) , a rare clonal malign

Missed entity:  'unilateral retinoblastoma'
Predicted as:    I-Disease
Sentence:        In most patients with isolated unilateral retinoblastoma , tumor development is initiated by somatic inactivation of bot

Missed entity:  'tumors'
Predicted as:    I-Disease
Sentence:        In 2 patients without a detectable mutation in peripheral blood , mosaicism was suggested because 1 of the patients show

Missed entity:  'glomerulonephritis'
Predicted as:    I-Disease
Sentence:        Of clinical interest are ( a ) the documentation of membranous glomerulonephritis , vasculitis , and arthritis in an ind

Missed entity:  'dominantly inherited neurodegeneration'
Predicted as:    O
Sentence:        This mutation may be valuable for developing models of dominantly inherited n

Pattern 1 — Predicted as I-Disease instead of B-Disease
Look at "unilateral retinoblastoma", "glomerulonephritis", "prostate cancer" — all predicted as I-Disease not O. This means the model detected something disease-like but got confused about the boundary. It thought these were continuations of a previous entity rather than the start of a new one. This is a boundary detection problem.
Pattern 2 — Complex multi-word entities missed entirely
"dominantly inherited neurodegeneration", "clonal malignancy", "congenital DM" — these are conceptually diseases but written in unusual ways. The model hasn't seen these exact phrasings enough during training.
Pattern 3 — Abbreviations and shorthand
"congenital DM" — DM is an abbreviation for Diabetes Mellitus. The model doesn't recognize abbreviated disease names well because BioBERT sees full words more often than abbreviations during pre-training.


In [10]:
short_missed = []
long_missed = []

for miss in missed_entities:
    length = len(miss["token_positions"])
    if length == 1:
        short_missed.append(miss)
    else:
        long_missed.append(miss)

print("=== Error Pattern Analysis ===\n")
print(f"Single word diseases missed:    {len(short_missed)}")
print(f"Multi word diseases missed:     {len(long_missed)}")
print(f"\nInsight: Model struggles more with {'single' if len(short_missed) > len(long_missed) else 'multi'}-word entities")

=== Error Pattern Analysis ===

Single word diseases missed:    30
Multi word diseases missed:     52

Insight: Model struggles more with multi-word entities


63% of all errors are multi-word entities. This is actually a known limitation of BIO tagging — the longer the entity, the more chances there are for one token to be mislabeled which breaks the whole entity span.

"Error analysis on the test set revealed that 63% of missed entities were multi-word disease mentions, suggesting the model struggles with long entity boundary detection. A secondary error pattern involved entities predicted as I-Disease rather than B-Disease, indicating boundary confusion rather than complete failure to detect the entity. Abbreviations and uncommon disease phrasings accounted for a third category of errors."